In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]

In [18]:
pa_gs_yr_gsl = pa_gs[['year', 'growing_season_length', 'station_id']]
pa_gs_non_gsl = pa_gs.drop(['growing_season_length'], axis = 1)

In [19]:
pa_gs_lagged = pa_gs_non_gsl.copy(deep=True)
pa_gs_lagged['year'] += 1

In [20]:
pa_gs_non_gsl.head(5)

,station_id,year,last_spring_frost_doy,last_spring_frost_date,first_fall_frost_doy,first_fall_frost_date,dtr_annual,dtr_spring,dtr_summer,tmax_annual,...,cloud_cover_pennsylvania,evaporation_pennsylvania,dewpoint_station,soil_moisture_station,cloud_cover_station,evaporation_station,tmean_spring,tmean_april,tmean_fall,tmean_october
0,USC00360022,2011,97,2011-04-07,301,2011-10-28,10.472299,11.314444,12.800000,16.367313,...,0.643602,1.986064,6.147552,0.457306,0.665281,2.143988,10.148333,10.916071,12.314286,11.000000
1,USC00360022,2012,88,2012-03-28,287,2012-10-13,9.794857,11.177174,9.884783,18.096571,...,0.594570,1.912246,5.980743,0.431833,0.587725,1.938779,13.512500,9.955000,12.115000,11.946774
2,USC00360022,2013,135,2013-05-15,301,2013-10-28,10.373088,11.328571,12.740244,15.384136,...,0.640574,1.798304,4.794067,0.441306,0.678640,1.894341,9.203846,10.683333,11.280220,12.712903
3,USC00360022,2014,115,2014-04-25,307,2014-11-03,9.648923,11.374419,10.137838,13.783692,...,0.645570,1.872335,4.093048,0.454242,0.673488,2.092208,8.120930,9.065000,10.852381,12.476667
4,USC00360022,2015,93,2015-04-03,292,2015-10-19,9.443731,10.935556,9.436471,16.000612,...,0.641373,1.805471,5.336334,0.437924,0.658836,1.886143,9.212222,10.750000,13.650610,11.162069


In [21]:
pa_gs_lagged.head(5)

,station_id,year,last_spring_frost_doy,last_spring_frost_date,first_fall_frost_doy,first_fall_frost_date,dtr_annual,dtr_spring,dtr_summer,tmax_annual,...,cloud_cover_pennsylvania,evaporation_pennsylvania,dewpoint_station,soil_moisture_station,cloud_cover_station,evaporation_station,tmean_spring,tmean_april,tmean_fall,tmean_october
0,USC00360022,2012,97,2011-04-07,301,2011-10-28,10.472299,11.314444,12.800000,16.367313,...,0.643602,1.986064,6.147552,0.457306,0.665281,2.143988,10.148333,10.916071,12.314286,11.000000
1,USC00360022,2013,88,2012-03-28,287,2012-10-13,9.794857,11.177174,9.884783,18.096571,...,0.594570,1.912246,5.980743,0.431833,0.587725,1.938779,13.512500,9.955000,12.115000,11.946774
2,USC00360022,2014,135,2013-05-15,301,2013-10-28,10.373088,11.328571,12.740244,15.384136,...,0.640574,1.798304,4.794067,0.441306,0.678640,1.894341,9.203846,10.683333,11.280220,12.712903
3,USC00360022,2015,115,2014-04-25,307,2014-11-03,9.648923,11.374419,10.137838,13.783692,...,0.645570,1.872335,4.093048,0.454242,0.673488,2.092208,8.120930,9.065000,10.852381,12.476667
4,USC00360022,2016,93,2015-04-03,292,2015-10-19,9.443731,10.935556,9.436471,16.000612,...,0.641373,1.805471,5.336334,0.437924,0.658836,1.886143,9.212222,10.750000,13.650610,11.162069


In [25]:
pa_gs_lagged_vars = pa_gs_lagged.merge(pa_gs_yr_gsl, on=["station_id", "year"], how = "left")

In [26]:
pa_gs_lagged_vars.head(5)

,station_id,year,last_spring_frost_doy,last_spring_frost_date,first_fall_frost_doy,first_fall_frost_date,dtr_annual,dtr_spring,dtr_summer,tmax_annual,...,evaporation_pennsylvania,dewpoint_station,soil_moisture_station,cloud_cover_station,evaporation_station,tmean_spring,tmean_april,tmean_fall,tmean_october,growing_season_length
0,USC00360022,2012,97,2011-04-07,301,2011-10-28,10.472299,11.314444,12.800000,16.367313,...,1.986064,6.147552,0.457306,0.665281,2.143988,10.148333,10.916071,12.314286,11.000000,199.0
1,USC00360022,2013,88,2012-03-28,287,2012-10-13,9.794857,11.177174,9.884783,18.096571,...,1.912246,5.980743,0.431833,0.587725,1.938779,13.512500,9.955000,12.115000,11.946774,166.0
2,USC00360022,2014,135,2013-05-15,301,2013-10-28,10.373088,11.328571,12.740244,15.384136,...,1.798304,4.794067,0.441306,0.678640,1.894341,9.203846,10.683333,11.280220,12.712903,192.0
3,USC00360022,2015,115,2014-04-25,307,2014-11-03,9.648923,11.374419,10.137838,13.783692,...,1.872335,4.093048,0.454242,0.673488,2.092208,8.120930,9.065000,10.852381,12.476667,199.0
4,USC00360022,2016,93,2015-04-03,292,2015-10-19,9.443731,10.935556,9.436471,16.000612,...,1.805471,5.336334,0.437924,0.658836,1.886143,9.212222,10.750000,13.650610,11.162069,212.0


In [38]:
def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string
    
def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid


all_covs_list = ['dtr_annual', 'dtr_spring', 'tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

In [39]:
lagged_model, lagged_output, lagged_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.121       31          142.55 draws/s       0:00:14    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.176       31          108.09 draws/s       0:00:18    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.150       31          137.20 draws/s       0:00:14    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.126       31          105.25 draws/s       0:00:18    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


In [40]:
az.summary(lagged_output)

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
Intercept,-137,77,-260,-15,2621,2760,1.00,1.5,1.2
dtr_annual,-11.99,0.49,-13,-11,2670,2942,1.00,0.0095,0.0068
dtr_spring,1.32,0.362,0.74,1.9,3716,3277,1.00,0.0059,0.0042
tmean_spring,0.01,0.302,-0.48,0.49,3210,2985,1.00,0.0053,0.0037
tmean_fall,3.262,0.303,2.8,3.7,4084,3365,1.00,0.0047,0.0034
latitude,-4.55,0.66,-5.6,-3.5,3707,3165,1.00,0.011,0.0076
longitude,1.965,0.222,1.6,2.3,3505,2970,1.00,0.0037,0.0026
tmax_annual,7.1,0.51,6.3,7.9,2901,3102,1.00,0.0095,0.0069
oni_annual,5.18,0.54,4.3,6,5000,3623,1.00,0.0076,0.0055
nao_annual,10.89,0.92,9.4,12,3836,3080,1.00,0.015,0.011


#### Analysis of Model (Notable Findings)
* inc of 1 degree to previous year annual DTR indicates 11.99 day decrease to current year GSL --> up from current year DTR coefficient of 9.51 day decrease
    * what would the logical cause of the ~2.5 unit increase in coefficient between current year DTR and lagged prior year DTR?
    * Is DTR predictive, or is this an artifact of general GSL increase year over year?
        * year over year GSL increase from main un-lagged model is 0.21 days/yr increase, so does not explain full 2.5 unit change to coefficient (let alone the fact that this year over year change in GSL would be going against the magnitude of the DTR influence
* inc of 1 degree to previous year annual DTR indicates 7.1329% increase to GSLV
* inc of 1 degree to previous year tmean spring indicates 5.096% increase to GSLV
* inc of 1 degree to previous year tmean fall indicates 1.9899% decrease to GSLV
* inc of 1 degree to previous year annual mean tmax indicates 7.263% decrease to GSLV
    * how does this relate to the finding on Annual DTR and GSLV? Does annual DTR increase only increase GSLV if the increase in DTR is found at the bottom of the DTR range, rather than the top?